# Customer Segmentation

Wholesale Customer Segmentation Assignment

In [20]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

CSV_PATH = "raw_wholesale_customers.csv"
OUT_PATH = "segmented_wholesale_customers.csv"
FEATURES = ["Fresh", "Milk", "Grocery",
            "Frozen", "Detergents_Paper", "Delicassen"]
RANDOM_STATE = 42
K = 5

In [3]:
# --------------------------------
# 1) Load dataset
# --------------------------------
df = pd.read_csv(CSV_PATH)
print("\n=== INITIAL SNAPSHOT ===")
print(df.head())


=== INITIAL SNAPSHOT ===
   Channel  Region  Fresh  Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0        2       3  12669  9656     7561     214              2674        1338
1        2       3   7057  9810     9568    1762              3293        1776
2        2       3   6353  8808     7684    2405              3516        7844
3        1       3  13265  1196     4221    6404               507        1788
4        2       3  22615  5410     7198    3915              1777        5185


In [4]:
# --------------------------------
# 2) Select features + IQR cap
# L3: clip extreme spend values before scaling — same idea as house/loan pipelines
# --------------------------------
X = df[FEATURES].copy()


In [5]:
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper


low_fresh,  high_fresh = iqr_fun(X["Fresh"])
low_milk,    high_milk = iqr_fun(X["Milk"])
low_grocery, high_grocery = iqr_fun(X["Grocery"])
low_frozen,  high_frozen = iqr_fun(X["Frozen"])
low_det,     high_det = iqr_fun(X["Detergents_Paper"])
low_deli,    high_deli = iqr_fun(X["Delicassen"])

X["Fresh"] = X["Fresh"].clip(lower=low_fresh,  upper=high_fresh)
X["Milk"] = X["Milk"].clip(lower=low_milk,    upper=high_milk)
X["Grocery"] = X["Grocery"].clip(lower=low_grocery, upper=high_grocery)
X["Frozen"] = X["Frozen"].clip(lower=low_frozen,  upper=high_frozen)
X["Detergents_Paper"] = X["Detergents_Paper"].clip(
    lower=low_det, upper=high_det)
X["Delicassen"] = X["Delicassen"].clip(lower=low_deli, upper=high_deli)

df[FEATURES] = X

print("\n=== IQR CAP SUMMARY FOR EACH FEATURE ===")
print(
    f"Fresh             low={low_fresh:.2f}  high={high_fresh:.2f}  |  after cap min={X['Fresh'].min():.2f}  max={X['Fresh'].max():.2f}")
print(
    f"Milk              low={low_milk:.2f}  high={high_milk:.2f}  |  after cap min={X['Milk'].min():.2f}  max={X['Milk'].max():.2f}")
print(
    f"Grocery           low={low_grocery:.2f}  high={high_grocery:.2f}  |  after cap min={X['Grocery'].min():.2f}  max={X['Grocery'].max():.2f}")
print(
    f"Frozen            low={low_frozen:.2f}  high={high_frozen:.2f}  |  after cap min={X['Frozen'].min():.2f}  max={X['Frozen'].max():.2f}")
print(
    f"Detergents_Paper  low={low_det:.2f}  high={high_det:.2f}  |  after cap min={X['Detergents_Paper'].min():.2f}  max={X['Detergents_Paper'].max():.2f}")
print(
    f"Delicassen        low={low_deli:.2f}  high={high_deli:.2f}  |  after cap min={X['Delicassen'].min():.2f}  max={X['Delicassen'].max():.2f}")

print("\n=== FEATURES HEAD (after IQR cap) ===")
print(X.head())


=== IQR CAP SUMMARY FOR EACH FEATURE ===
Fresh             low=-17581.25  high=37642.75  |  after cap min=3.00  max=37642.75
Milk              low=-6952.88  high=15676.12  |  after cap min=55.00  max=15676.12
Grocery           low=-10601.12  high=23409.88  |  after cap min=3.00  max=23409.88
Frozen            low=-3475.75  high=7772.25  |  after cap min=25.00  max=7772.25
Detergents_Paper  low=-5241.12  high=9419.88  |  after cap min=3.00  max=9419.88
Delicassen        low=-1709.75  high=3938.25  |  after cap min=3.00  max=3938.25

=== FEATURES HEAD (after IQR cap) ===
     Fresh    Milk  Grocery  Frozen  Detergents_Paper  Delicassen
0  12669.0  9656.0   7561.0   214.0            2674.0     1338.00
1   7057.0  9810.0   9568.0  1762.0            3293.0     1776.00
2   6353.0  8808.0   7684.0  2405.0            3516.0     3938.25
3  13265.0  1196.0   4221.0  6404.0             507.0     1788.00
4  22615.0  5410.0   7198.0  3915.0            1777.0     3938.25


In [6]:
# --------------------------------
# 3) Scale features
# --------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("\nScaled shape:", X_scaled.shape)


Scaled shape: (440, 6)


In [7]:
# --------------------------------
# 5) Fit K-Means with chosen k
# --------------------------------
kmeans = KMeans(n_clusters=K, n_init="auto", random_state=RANDOM_STATE)
labels = kmeans.fit_predict(X_scaled)

df["Cluster"] = labels.astype(int)
print("\n=== SAMPLE WITH CLUSTERS ===")
print(df.head())


=== SAMPLE WITH CLUSTERS ===
   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   
3        1       3  13265.0  1196.0   4221.0  6404.0             507.0   
4        2       3  22615.0  5410.0   7198.0  3915.0            1777.0   

   Delicassen  Cluster  
0     1338.00        0  
1     1776.00        0  
2     3938.25        0  
3     1788.00        3  
4     3938.25        3  


In [9]:

# 6) Evaluate clustering
# --------------------------------
sil = silhouette_score(X_scaled, labels)
dbi = davies_bouldin_score(X_scaled, labels)

print("\n=== K-MEANS METRICS ===")
print(f"Silhouette Score : {sil:.3f}")
print(f"Davies-Bouldin Index : {dbi:.3f}")


=== K-MEANS METRICS ===
Silhouette Score : 0.283
Davies-Bouldin Index : 1.270


In [10]:
# --------------------------------
# 7) Cluster centers (Original Units)
# --------------------------------
centers_scaled = kmeans.cluster_centers_

centers_original = scaler.inverse_transform(
    centers_scaled
)

centers_df = pd.DataFrame(
    centers_original,
    columns=FEATURES
)

centers_df.index.name = "Cluster"

print("\n=== CLUSTER CENTERS (Original Units) ===")
print(centers_df.round(2))


=== CLUSTER CENTERS (Original Units) ===
            Fresh      Milk   Grocery   Frozen  Detergents_Paper  Delicassen
Cluster                                                                     
0         9202.67   6833.30   9104.12  1326.16           3280.12     1871.76
1         8376.23   2150.65   3160.63  1646.33            779.25      674.02
2        17461.54  13805.60  17524.12  4120.57           5460.56     3583.64
3        22346.70   3409.14   3969.33  5819.60            583.07     1566.95
4         4916.98  10768.85  18350.13  1212.37           7780.02      981.37


## Research & Train a Second Clustering Algorithm

For the second clustering algorithm, I selected **Agglomerative Clustering**.

I chose this algorithm because it builds clusters by gradually merging similar observations instead of using randomly initialized centroids like K-Means. This makes it a useful method for comparing different clustering approaches on the wholesale customer dataset.

### Research Source

Scikit-learn Documentation:

https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html

In [16]:
# --------------------------------
# 8) Train Agglomerative Clustering
# --------------------------------
agg = AgglomerativeClustering(
    n_clusters=5
)

agg_labels = agg.fit_predict(X_scaled)

df["Agglomerative_Cluster"] = agg_labels

print("\n=== SAMPLE WITH AGGLOMERATIVE CLUSTERS ===")
print(df.head())

agg_sil = silhouette_score(
    X_scaled,
    agg_labels
)

print(f"Agglomerative Silhouette Score: {agg_sil:.3f}")


=== SAMPLE WITH AGGLOMERATIVE CLUSTERS ===
   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   
3        1       3  13265.0  1196.0   4221.0  6404.0             507.0   
4        2       3  22615.0  5410.0   7198.0  3915.0            1777.0   

   Delicassen  Cluster  Agglomerative_Cluster  
0     1338.00        0                      4  
1     1776.00        0                      4  
2     3938.25        0                      0  
3     1788.00        3                      3  
4     3938.25        3                      0  
Agglomerative Silhouette Score: 0.218


In [17]:
# --------------------------------
# 9) Compare Methods
# --------------------------------
print("\n=== COMPARISON ===")

print(f"K-Means Silhouette Score        : {sil:.3f}")
print(f"Agglomerative Silhouette Score : {agg_sil:.3f}")

if sil > agg_sil:
    print("\nK-Means produced better-separated clusters.")
elif agg_sil > sil:
    print("\nAgglomerative Clustering produced better-separated clusters.")
else:
    print("\nBoth methods produced similar clustering quality.")


=== COMPARISON ===
K-Means Silhouette Score        : 0.283
Agglomerative Silhouette Score : 0.218

K-Means produced better-separated clusters.


In [18]:
# --------------------------------
# 10) Sanity Check (3 Clients)
# --------------------------------
sample_idx = [0, 1, 2]

sanity = df.loc[
    sample_idx,
    [
        "Channel",
        "Region",
        "Fresh",
        "Milk",
        "Grocery",
        "Frozen",
        "Detergents_Paper",
        "Delicassen",
        "Cluster",
        "Agglomerative_Cluster"
    ]
]

print("\n=== SANITY CHECK (3 CLIENTS) ===")
print(sanity)


=== SANITY CHECK (3 CLIENTS) ===
   Channel  Region    Fresh    Milk  Grocery  Frozen  Detergents_Paper  \
0        2       3  12669.0  9656.0   7561.0   214.0            2674.0   
1        2       3   7057.0  9810.0   9568.0  1762.0            3293.0   
2        2       3   6353.0  8808.0   7684.0  2405.0            3516.0   

   Delicassen  Cluster  Agglomerative_Cluster  
0     1338.00        0                      4  
1     1776.00        0                      4  
2     3938.25        0                      0  


In [22]:
# --------------------------------
# 11) Save Dataset
# --------------------------------
OUT_PATH = "segmented_wholesale_customers.csv"

df.to_csv(
    OUT_PATH,
    index=False
)

print(f"\nDataset saved successfully -> {OUT_PATH}")


Dataset saved successfully -> segmented_wholesale_customers.csv


## Research & Train a Second Clustering Algorithm

I selected **Agglomerative Clustering** because it builds clusters by gradually merging similar customers and provides a useful comparison with K-Means.

**Research Source:** https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html